# P4-S5 — Bielik LoRA/QLoRA fine-tuning (generative decoder)

Fine-tunes `speakleash/Bielik-1.5B-v3.0-Instruct` (Apache 2.0, verified live on the model card 2026-09-19) with LoRA/QLoRA (`peft` + `trl` + `bitsandbytes`) into a generative labeler for the same task as the teacher/HerBERT: given a headline, output the `Label` JSON (`sentiment`, `event_type`, `tickers`).

Unlike HerBERT (a classification head with a fixed label space baked into the model), this is a plain text generator — nothing stops it from producing invalid JSON or an out-of-enum value. REQ-012 applies here directly: an unparseable or schema-invalid response is recorded as a labeling error with a category, never guessed into a partial label.

**Runtime:** `Runtime` → `Change runtime type` → `T4 GPU`, before running any cell below.

**Caveat (2026-09-19):** same 92-headline corpus as `herbert_finetune.ipynb` (86 after dedup) — this is a pipeline sanity check, not a real quality signal (see `docs/ROADMAP.md`).

**Zero cost:** no LLM API calls — only local (Colab GPU) training. CLAUDE.md rule 9 doesn't apply here.

**Library APIs drift fast** (same lesson as `herbert_finetune.ipynb`'s `processing_class=`/`labels=` fixes) — `peft`/`trl` on an unpinned Colab install may need small fixes once run live.

## 1. Clone the repo and install

In [ ]:
import os
import sys

# Same idempotent-clone fix as herbert_finetune.ipynb — a relative `%cd`
# run twice nests a second clone inside the first.
REPO_DIR = "/content/fin-ai-lab"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/devTomaszStoklosa/fin-ai-lab.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -e . -q
!pip install transformers peft trl bitsandbytes accelerate datasets -q

# Same reason as herbert_finetune.ipynb: an editable install's .pth file
# is only read at interpreter startup, not mid-session.
SRC_DIR = f"{REPO_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)


## 2. Get the corpus onto this Colab session

Same corpus as `herbert_finetune.ipynb` — if you already uploaded it to Drive for that notebook, point `CORPUS_PATH` at the same file.

1. Upload `data/corpus/news_classifier/labeled.jsonl` to your Google Drive, e.g. `My Drive/fin-ai-lab/labeled.jsonl`.
2. Update `CORPUS_PATH` below to match where you put it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CORPUS_PATH = "/content/drive/MyDrive/fin-ai-lab/labeled.jsonl"
CHECKPOINT_DIR = "/content/drive/MyDrive/fin-ai-lab/checkpoints/bielik-lora"


## 3. Load and split the corpus

Same `corpus_store`/`split` code as everywhere else in P4 — same chronological split, same near-duplicate dedup (REQ-005).

In [ ]:
from pathlib import Path

from fin_ai_lab.news_classifier.corpus_store import load_labeled
from fin_ai_lab.news_classifier.split import split_chronological

corpus = load_labeled(Path(CORPUS_PATH))
print(f"Loaded {len(corpus)} labeled headlines")

train_items, dev_items, test_items = split_chronological(corpus)
print(f"train={len(train_items)} dev={len(dev_items)} test={len(test_items)}")


## 4. Build the training prompt

Reuses `prompts/bielik_sft.v1.md` (`core.prompts.PromptRegistry`, same versioned-prompt setup as the teacher pipeline — ADR 0004) rather than hardcoding the instruction text here. It's a separate prompt file from `teacher.v2.md`, not a shared one: `teacher.v2` relies on `response_schema` (Gemini/Groq structured output) to enforce the JSON *shape*, but this model has no such constraint on plain-text generation, so `bielik_sft.v1` spells the JSON shape out in the prompt text too — the same "the model only ever sees the prompt text" lesson `teacher.v2` already taught for the enum list.

Training text = the rendered prompt (user turn) + the ground-truth `Label` JSON (assistant turn), formatted through the model's own chat template — never a hand-rolled format that might not match what the tokenizer/model were trained to expect.

In [ ]:
import json

from transformers import AutoTokenizer

from fin_ai_lab.core.prompts.registry import PromptRegistry
from fin_ai_lab.news_classifier.models import LabeledHeadline
from fin_ai_lab.news_classifier.ticker_catalog import TICKER_CATALOG

MODEL_ID = "speakleash/Bielik-1.5B-v3.0-Instruct"

prompts = PromptRegistry()
prompts.load_dir(Path("src/fin_ai_lab/news_classifier/prompts"))
sft_prompt = prompts.get("bielik_sft", 1)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
TICKER_CATALOG_JSON = json.dumps(TICKER_CATALOG, ensure_ascii=False)


def render_user_prompt(item: LabeledHeadline) -> str:
    return sft_prompt.render(
        headline=item.headline.headline,
        lead=item.headline.lead or "",
        ticker_catalog_json=TICKER_CATALOG_JSON,
    )


def target_json(item: LabeledHeadline) -> str:
    return json.dumps(
        {
            "sentiment": item.label.sentiment,
            "event_type": item.label.event_type,
            "tickers": item.label.tickers,
        },
        ensure_ascii=False,
    )


def training_text(item: LabeledHeadline) -> str:
    messages = [
        {"role": "user", "content": render_user_prompt(item)},
        {"role": "assistant", "content": target_json(item)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)


def generation_prompt(item: LabeledHeadline) -> str:
    messages = [{"role": "user", "content": render_user_prompt(item)}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


## 5. Load the base model in 4-bit (QLoRA) and wrap it with LoRA

`get_peft_model` is called explicitly here (rather than handed to `SFTTrainer` as a `peft_config`) so the LoRA wiring itself is visible in this notebook's own code, not hidden inside a library call — that's the point of this slice.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Standard LLaMA-style attention/MLP projections -- Bielik's own model
    # card describes a LLaMA-like architecture (verified live 2026-09-19).
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


## 6. Fine-tune with TRL's SFTTrainer

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_dict({"text": [training_text(item) for item in train_items]})
dev_ds = Dataset.from_dict({"text": [training_text(item) for item in dev_items]})

sft_config = SFTConfig(
    output_dir="/content/bielik-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=5,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=512,
)
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
)
trainer.train()


## 7. Evaluate on the held-out test split

Generates a completion per test headline, then tries to parse it as JSON and validate it against the `Label` schema. A response that fails either check is a labeling error with a category (REQ-012) — never guessed into a partial label, and never silently dropped from the count either (docs/EVALS.md rule 5: report the failure, don't hide it in an average).

Reuses `news_classifier/metrics.py` (built for P4-S7's comparison report) for macro-F1/confusion matrix — same metric code as every other compared model, not reimplemented here.

In [ ]:
import json as json_module
from typing import get_args

from pydantic import ValidationError

from fin_ai_lab.news_classifier.metrics import confusion_matrix, macro_f1
from fin_ai_lab.news_classifier.models import EventType, Label, Sentiment

SENTIMENTS = list(get_args(Sentiment))
EVENT_TYPES = list(get_args(EventType))

model.eval()


def generate_label(item: LabeledHeadline) -> Label | None:
    prompt_text = generation_prompt(item)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    completion = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    try:
        parsed = json_module.loads(completion)
        return Label.model_validate(parsed)
    except (json_module.JSONDecodeError, ValidationError):
        return None


sentiment_pairs = []
event_type_pairs = []
schema_errors = 0

for item in test_items:
    predicted = generate_label(item)
    if predicted is None:
        schema_errors += 1
        continue
    sentiment_pairs.append((item.label.sentiment, predicted.sentiment))
    event_type_pairs.append((item.label.event_type, predicted.event_type))

print(f"schema_errors: {schema_errors} / {len(test_items)} (REQ-012)")
if sentiment_pairs:
    print(f"sentiment macro-F1: {macro_f1(sentiment_pairs, SENTIMENTS):.3f}")
    print(f"event_type macro-F1: {macro_f1(event_type_pairs, EVENT_TYPES):.3f}")
    print("sentiment confusion matrix:", confusion_matrix(sentiment_pairs, SENTIMENTS))
else:
    print("Every test item failed schema validation -- nothing to score.")


## 8. Save the LoRA adapter to Drive

`save_pretrained` on a PEFT model saves only the adapter weights (a few MB), not the base model — never committed to git (`checkpoints/` gitignored), stays on Drive like the HerBERT checkpoints.

In [ ]:
model.save_pretrained(CHECKPOINT_DIR)
print(f"Saved adapter to {CHECKPOINT_DIR}")


## Next steps

- P4-S6: quantize a fine-tuned model to GGUF and measure real local-CPU latency (`docs/ENVIRONMENT.md` — no AVX2, no GPU on this machine).
- P4-S7: extend `qa_target.py`'s comparison report with this model alongside majority/TF-IDF/few-shot/HerBERT.